# GeoFWI3D — UNETR Dice + 2CE, then conditional BANDS
One fresh fold-0 / seed-0 baseline. Change CE weight from 1 to **2**; keep ViT-B/16,
XY D4 augmentation, dropout 0.1, LR 1e-4, AdamW decay 1e-5, fixed 477-update LR horizon,
BF16, batch 1, native 96³ and identical initialization. 19 train / 96 validation volumes.

**Cap per arm: 182,178.12654316 GFLOPs. Baseline: 414 updates.**
Stop at 207 unless the mean original validation Dice at updates 165 and 207 is **>=46%**.
Success requires **final-budget validation Dice strictly >50%**. Best Dice never triggers BANDS.

If and only if the verified archived baseline passes, automatically run **one matched BANDS arm**
from the same fresh seed-0 initialization. Same Dice+2CE base recipe and training sequence;
50 shared base-loss updates, then training-only gradient calibration (target 0.10, p95 cap 0.50)
and a 50-update BANDS ramp. Two-voxel six-connected inner/outer salt boundaries.
BANDS fits **407 updates including its measured calibration within the same cap**.
The same 46% halfway gate applies. A pruned BANDS arm is explicitly marked as an incomplete full-budget comparison.
At most two caps in total; no automatic further seeds or recipes.

Three GPU preflight updates per launched arm are diagnostic and separately reported, as in the established convention.
Validation/setup are outside optimization accounting. Archive verification precedes checkpoint retirement.
Interruptions preserve recovery; reruns never repeat completed stages. This is a development experiment, not the 3×3 baseline.
Account **focacciafilippo@gmail.com**, **A100-SXM4-80GB + High RAM**. Run one copy.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'focacciafilippo@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_unetr_ce_pilot_20261019')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_unetr_ce2_bands_f0s0_v1'
BUNDLE=Path('/content/hippo_geofwi_unetr_ce2_bands_v1')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 24*1024**3, 'Need at least 24 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


## Verify and stage the uploaded data/code
All archive and per-file SHA256 hashes are checked. The 319 unique images/labels are read from
the runtime SSD during training. Existing matching files are reused; changed files stop execution.


In [ ]:
EXPECTED_UPLOADS={'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880', 'geofwi3d_unetr_ce2_bands_code_v1.zip': '310a21d9a020673699839ab105b240b148bc3a50b69f0eeab0c8e470bb798716'}
EXPECTED_MANIFEST='2ecfeca4c23a0ecd0c48a834f6a0ff9b038f57ee5036e8bc1bf85e90997ec2ca'
existing=Path('/content/hippo_geofwi_unetr_xy_aug_v1/data')
if existing.is_dir() and not (BUNDLE/'data').exists():
    (BUNDLE/'data').symlink_to(existing,target_is_directory=True)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    if not package.exists():raise FileNotFoundError(str(package))
    assert sha(package)==expected, 'Upload checksum mismatch: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:
        shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist())), 'Duplicate archive members'
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
        for info in z.infolist():
            target=BUNDLE/info.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            temp.replace(target)
    print('Staged',name)
assert sha(BUNDLE/'bundle_manifest.json')==EXPECTED_MANIFEST
for listing in ['bundle_manifest.json','data_manifest.json']:
    for name,expected in json.loads((BUNDLE/listing).read_text()).items():
        assert sha(BUNDLE/name)==expected, 'Extracted file mismatch: '+name
source=RESULTS/'provenance'/EXPECTED_MANIFEST;source.mkdir(parents=True,exist_ok=True)
for name in ['geofwi3d_unetr_ce2_bands_code_v1.zip']:
    dst=source/name
    if not dst.exists():shutil.copyfile(INPUT/name,dst)
    assert sha(dst)==EXPECTED_UPLOADS[name]
print('All inputs verified. Training augmentation: XY D4 only.')


## Install the frozen environment
Uses the package versions from the existing Colab training environment, in a separate venv.
Setup output is streamed and saved in Drive. Keep the same GPU type/software on resume.


In [ ]:
VENV=Path('/content/hippo_geofwi_unetr_ce_pilot_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'],'setup.log')
def campaign(action,*extra):
    command([PYTHON,'-u',BUNDLE/'ce2_runtime.py',action,'--bundle',BUNDLE,'--durable',RESULTS,*extra],action+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Required GPU checks
Native augmented forward/backward, finite gradients and exact model/optimizer/RNG replay. Conditional BANDS is checked only after baseline success.


In [ ]:
campaign('preflight')


## Run or resume
Apply the 46% halfway gate; finish the baseline only if it passes. Automatically launch matched BANDS only for verified final baseline Dice >50%.


In [ ]:
SESSION_HOURS=2.0
# Graceful stop: create RESULTS/'STOP'. Rerun to resume a timed pause.
campaign('run','--hours',str(SESSION_HOURS))


## Results
`ce2_bands_status.json` separates baseline qualification, conditional BANDS and technical pauses. `EXPERIMENT_FINISHED.json` records a completed or pruned job.


In [ ]:
campaign('report')
status=json.loads((RESULTS/'ce2_bands_status.json').read_text())
print(json.dumps(status,indent=2))
